# Agglomerative (Hierarchical) Clustering on Retail Data

**Objective:** Discover customer segments using bottom-up hierarchical clustering and visualize with a dendrogram.

**Pipeline:** Load Data → EDA → Scaling → Dendrogram → Linkage Methods → Agglomerative Clustering → Evaluation

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

from retail_data import generate_retail_dataset

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

## 1. Load Retail Dataset

In [ ]:
df = generate_retail_dataset(n_samples=2000, random_state=RANDOM_STATE)
print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
numerical_cols = ['Age', 'Annual_Income', 'Spending_Score', 'Num_Purchases', 'Avg_Transaction_Value', 'Total_Sales']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='Annual_Income', y='Spending_Score', hue='Customer_Segment', ax=axes[0], palette='Set1', alpha=0.6)
axes[0].set_title('Income vs Spending by Segment')
sns.scatterplot(data=df, x='Num_Purchases', y='Total_Sales', hue='Customer_Segment', ax=axes[1], palette='Set1', alpha=0.6)
axes[1].set_title('Purchases vs Sales by Segment')
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
feature_cols = ['Age', 'Annual_Income', 'Spending_Score', 'Num_Purchases', 'Avg_Transaction_Value']
X = df[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Scaled data shape: {X_scaled.shape}')

## 4. Dendrogram Visualization

We use a sample for readability (full 2000-point dendrogram is too dense).

In [ ]:
sample_size = 150
sample_idx = np.random.default_rng(RANDOM_STATE).choice(len(X_scaled), size=sample_size, replace=False)
X_sample = X_scaled[sample_idx]

linkage_methods = ['ward', 'complete', 'average']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, method in zip(axes, linkage_methods):
    Z = linkage(X_sample, method=method)
    dendrogram(Z, ax=ax, truncate_mode='lastp', p=12, leaf_rotation=45)
    ax.set_title(f'Dendrogram ({method} linkage)')
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Distance')

plt.tight_layout()
plt.show()

## 5. Compare Linkage Methods & Optimal K

In [ ]:
k_range = range(2, 9)
results = []

for method in ['ward', 'complete', 'average']:
    for k in k_range:
        model = AgglomerativeClustering(n_clusters=k, linkage=method)
        labels = model.fit_predict(X_scaled)
        sil = silhouette_score(X_scaled, labels)
        db = davies_bouldin_score(X_scaled, labels)
        results.append({'linkage': method, 'k': k, 'silhouette': sil, 'davies_bouldin': db})

results_df = pd.DataFrame(results)
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for method in ['ward', 'complete', 'average']:
    subset = results_df[results_df['linkage'] == method]
    axes[0].plot(subset['k'], subset['silhouette'], 'o-', label=method)
    axes[1].plot(subset['k'], subset['davies_bouldin'], 'o-', label=method)

axes[0].set_xlabel('K'); axes[0].set_ylabel('Silhouette'); axes[0].set_title('Silhouette by K')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Davies-Bouldin'); axes[1].set_title('Davies-Bouldin by K')
axes[0].legend(); axes[1].legend()
plt.tight_layout()
plt.show()

best = results_df.loc[results_df['silhouette'].idxmax()]
print(f'Best config: linkage={best["linkage"]}, K={int(best["k"])}, silhouette={best["silhouette"]:.4f}')

## 6. Train Final Agglomerative Model

In [ ]:
OPTIMAL_K = int(best['k'])
BEST_LINKAGE = best['linkage']

agg = AgglomerativeClustering(n_clusters=OPTIMAL_K, linkage=BEST_LINKAGE)
df['Cluster'] = agg.fit_predict(X_scaled)

print('Cluster distribution:')
print(df['Cluster'].value_counts().sort_index())
print(f'Silhouette: {silhouette_score(X_scaled, df["Cluster"]):.4f}')

## 7. Visualize Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='Annual_Income', y='Spending_Score', hue='Cluster', palette='tab10', ax=axes[0], s=50, alpha=0.7)
axes[0].set_title(f'Agglomerative Clusters (K={OPTIMAL_K}, {BEST_LINKAGE})')
sns.scatterplot(data=df, x='Num_Purchases', y='Total_Sales', hue='Cluster', palette='tab10', ax=axes[1], s=50, alpha=0.7)
axes[1].set_title('Purchases vs Total Sales')
plt.tight_layout()
plt.show()

In [ ]:
cluster_means = df.groupby('Cluster')[feature_cols + ['Total_Sales']].mean().round(2)
cluster_means

In [ ]:
pd.crosstab(df['Cluster'], df['Customer_Segment'], normalize='index').round(3)

## 8. Conclusions

- **Agglomerative clustering** builds a hierarchy of merges — useful when you want to explore cluster structure at multiple granularities.
- The **dendrogram** helps choose K by cutting at a distance threshold.
- **Ward linkage** typically works well for compact, similarly-sized clusters on scaled numerical retail features.
- **Limitation:** O(n²) memory/time complexity — use sampling for dendrograms on large datasets.